# NeuroState Qwen Causal Intervention v0.5

Builds a Qwen2.5-0.5B proceed/hesitate-specific direction, then compares early layer 7 with the strongest v0.4 site at layer 13.

Select a T4 GPU runtime, then run all cells.

In [ ]:
!pip -q install 'torch>=2.2' 'transformers>=4.40' 'accelerate>=0.28'


In [ ]:
from pathlib import Path
WORK=Path('/content/neurostate_qwen_causal_intervention_v0_5')
WORK.mkdir(parents=True,exist_ok=True)


In [ ]:
(WORK/'intervention_core.py').write_text('from __future__ import annotations\n\nimport math\nimport random\nfrom typing import Sequence\n\nPROMPTS = (\n    "Write one short sentence about opening a window.",\n    "Write one short sentence about preparing tea.",\n    "Write one short sentence about an empty notebook.",\n    "Write one short sentence about walking to a station.",\n    "Write one short sentence about rain on a roof.",\n    "Write one short sentence about arranging a desk.",\n    "Write one short sentence about a quiet library.",\n    "Write one short sentence about starting a small task.",\n    "Write one short sentence about watching clouds.",\n    "Write one short sentence about tying a shoelace.",\n    "Write one short sentence about a lamp at dusk.",\n    "Write one short sentence about washing a cup.",\n    "Write one short sentence about reading a map.",\n    "Write one short sentence about hearing distant traffic.",\n    "Write one short sentence about folding a towel.",\n    "Write one short sentence about checking the time.",\n    "Write one short sentence about a garden path.",\n    "Write one short sentence about closing a drawer.",\n    "Write one short sentence about sharpening a pencil.",\n    "Write one short sentence about waiting for an elevator.",\n)\n\nACTIVE_WORDS = (" move", " act", " forward", " begin", " go", " start")\nCALM_WORDS = (" rest", " calm", " quiet", " wait", " pause", " still")\nPROCEED_WORDS = (" proceed", " continue", " advance", " move", " begin", " start")\nHESITATE_WORDS = (" hesitate", " delay", " pause", " wait", " stop", " hold")\n\nCONTRAST_BANKS = {\n    "action_calm": (ACTIVE_WORDS, CALM_WORDS),\n    "proceed_hesitate": (PROCEED_WORDS, HESITATE_WORDS),\n}\n\n\ndef dot(a: Sequence[float], b: Sequence[float]) -> float:\n    return sum(x * y for x, y in zip(a, b))\n\n\ndef norm(vector: Sequence[float]) -> float:\n    return math.sqrt(dot(vector, vector))\n\n\ndef normalize(vector: Sequence[float]) -> list[float]:\n    length = norm(vector)\n    if not length:\n        raise ValueError("cannot normalize zero vector")\n    return [value / length for value in vector]\n\n\ndef orthogonal_random_direction(reference: Sequence[float], seed: int) -> list[float]:\n    unit_reference = normalize(reference)\n    rng = random.Random(seed)\n    candidate = [rng.gauss(0.0, 1.0) for _ in unit_reference]\n    projection = dot(candidate, unit_reference)\n    orthogonal = [value - projection * axis for value, axis in zip(candidate, unit_reference)]\n    return normalize(orthogonal)\n\n\ndef linear_slope(xs: Sequence[float], ys: Sequence[float]) -> float:\n    x_mean = sum(xs) / len(xs)\n    y_mean = sum(ys) / len(ys)\n    denominator = sum((x - x_mean) ** 2 for x in xs)\n    if denominator == 0:\n        raise ValueError("slope requires varying x")\n    return sum((x - x_mean) * (y - y_mean) for x, y in zip(xs, ys)) / denominator\n\n\ndef exact_sign_flip_p(values: Sequence[float]) -> float:\n    observed = abs(sum(values) / len(values))\n    exceed = 0\n    total = 1 << len(values)\n    for mask in range(total):\n        permuted = sum((-value if mask & (1 << i) else value) for i, value in enumerate(values)) / len(values)\n        exceed += abs(permuted) >= observed - 1e-12\n    return exceed / total\n\n\ndef keyword_counts(text: str, positive_words: Sequence[str], negative_words: Sequence[str]) -> dict[str, int]:\n    lowered = " " + text.lower()\n    return {\n        "positive_word_count": sum(lowered.count(word) for word in positive_words),\n        "negative_word_count": sum(lowered.count(word) for word in negative_words),\n    }\n',encoding='utf-8')


In [ ]:
(WORK/'build_direction.py').write_text('from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nMODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"\n\nCONTRAST_CONTEXTS = (\n    "opening a window",\n    "preparing tea",\n    "starting a small task",\n    "checking the time",\n    "arranging a desk",\n    "walking to a station",\n    "reading a map",\n    "folding a towel",\n    "waiting near an elevator",\n    "writing in an empty notebook",\n    "hearing distant traffic",\n    "closing a drawer",\n)\n\nPROCEED_FRAMES = (\n    "The response should proceed and continue with the next step about {context}.",\n    "The response should begin and move forward with the next step about {context}.",\n)\n\nHESITATE_FRAMES = (\n    "The response should hesitate and delay the next step about {context}.",\n    "The response should pause and wait before the next step about {context}.",\n)\n\n\ndef paired_prompts() -> list[tuple[str, str]]:\n    pairs = []\n    for context in CONTRAST_CONTEXTS:\n        for proceed, hesitate in zip(PROCEED_FRAMES, HESITATE_FRAMES):\n            pairs.append((proceed.format(context=context), hesitate.format(context=context)))\n    return pairs\n\n\ndef decoder_layers(model):\n    if hasattr(model, "model") and hasattr(model.model, "layers"):\n        return model.model.layers\n    raise ValueError("unsupported decoder layout")\n\n\ndef subtract(left: list[float], right: list[float]) -> list[float]:\n    return [a - b for a, b in zip(left, right)]\n\n\ndef mean_vector(vectors: list[list[float]]) -> list[float]:\n    if not vectors:\n        raise ValueError("cannot average an empty vector list")\n    width = len(vectors[0])\n    if any(len(vector) != width for vector in vectors):\n        raise ValueError("all vectors must have the same dimension")\n    return [sum(vector[index] for vector in vectors) / len(vectors) for index in range(width)]\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--target-layer", type=int, default=13)\n    parser.add_argument("--output", type=Path, required=True)\n    args = parser.parse_args()\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device == "cuda" else torch.float32\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_NAME,\n        torch_dtype=dtype,\n        device_map="auto" if device == "cuda" else None,\n    )\n    if device == "cpu":\n        model.to(device)\n    model.eval()\n\n    hidden_state_index = args.target_layer + 1\n    deltas = []\n    proceed_vectors = []\n    hesitate_vectors = []\n    for pair_index, (proceed_text, hesitate_text) in enumerate(paired_prompts()):\n        vectors = []\n        for text in (proceed_text, hesitate_text):\n            inputs = tokenizer(text, return_tensors="pt").to(model.device)\n            with torch.inference_mode():\n                result = model(**inputs, use_cache=False, output_hidden_states=True, return_dict=True)\n            vector = result.hidden_states[hidden_state_index][0, -1, :].float().cpu().tolist()\n            vectors.append(vector)\n        proceed_vector, hesitate_vector = vectors\n        proceed_vectors.append(proceed_vector)\n        hesitate_vectors.append(hesitate_vector)\n        deltas.append(subtract(proceed_vector, hesitate_vector))\n        print(f"{pair_index + 1:02d} layer={args.target_layer} built delta")\n\n    raw_delta = mean_vector(deltas)\n    mean_proceed = mean_vector(proceed_vectors)\n    mean_hesitate = mean_vector(hesitate_vectors)\n    raw_delta_norm = sum(value * value for value in raw_delta) ** 0.5\n    metadata = {\n        "source": "proceed_hesitate_paired_prompts_v0_5",\n        "model": MODEL_NAME,\n        "field": "proceed_hesitate_last_hidden_delta",\n        "pair_count": len(deltas),\n        "dimension": len(raw_delta),\n        "hidden_state_index": hidden_state_index,\n        "decoder_layer_index": args.target_layer,\n        "raw_delta_norm": raw_delta_norm,\n        "mean_proceed_norm": sum(value * value for value in mean_proceed) ** 0.5,\n        "mean_hesitate_norm": sum(value * value for value in mean_hesitate) ** 0.5,\n        "positive_label": "proceed",\n        "negative_label": "hesitate",\n        "raw_delta": raw_delta,\n    }\n    args.output.write_text(json.dumps(metadata, ensure_ascii=False) + "\\n", encoding="utf-8")\n    print(args.output)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
(WORK/'run_intervention.py').write_text('from __future__ import annotations\n\nimport argparse\nimport json\nfrom pathlib import Path\n\nfrom intervention_core import (\n    ACTIVE_WORDS,\n    CALM_WORDS,\n    CONTRAST_BANKS,\n    PROMPTS,\n    keyword_counts,\n    normalize,\n    orthogonal_random_direction,\n)\n\nMODEL_NAME = "Qwen/Qwen2.5-0.5B-Instruct"\nSEMANTIC_ALPHAS = (-2.0, -1.0, -0.5, 0.0, 0.5, 1.0, 2.0)\nRANDOM_ALPHAS = (-2.0, 2.0)\n\n\ndef decoder_layers(model):\n    if hasattr(model, "model") and hasattr(model.model, "layers"):\n        return model.model.layers\n    raise ValueError("unsupported decoder layout")\n\n\nclass SteeringHook:\n    def __init__(self, torch_module, direction: list[float], alpha: float):\n        self.torch = torch_module\n        self.direction = direction\n        self.alpha = alpha\n\n    def __call__(self, module, inputs, output):\n        hidden = output[0] if isinstance(output, tuple) else output\n        direction = self.torch.tensor(self.direction, device=hidden.device, dtype=hidden.dtype)\n        edited = hidden.clone()\n        edited[:, -1, :] = edited[:, -1, :] + self.alpha * direction\n        return (edited, *output[1:]) if isinstance(output, tuple) else edited\n\n\ndef token_ids(tokenizer, words: tuple[str, ...]) -> list[int]:\n    ids = []\n    for word in words:\n        encoded = tokenizer.encode(word, add_special_tokens=False)\n        if encoded:\n            ids.append(int(encoded[0]))\n    return sorted(set(ids))\n\n\ndef parse_int_list(raw: str | None) -> list[int]:\n    if not raw:\n        return []\n    values = []\n    for part in raw.split(","):\n        part = part.strip()\n        if part:\n            values.append(int(part))\n    return values\n\n\ndef contrast_fields(bank_name: str) -> tuple[str, str, str]:\n    return (\n        f"logit_contrast_{bank_name}",\n        f"positive_word_count_{bank_name}",\n        f"negative_word_count_{bank_name}",\n    )\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--direction-json", type=Path, required=True)\n    parser.add_argument("--mode", choices=("smoke", "full", "layer_scan"), default="smoke")\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--prompt-count", type=int, default=None)\n    parser.add_argument("--random-count", type=int, default=None)\n    parser.add_argument("--layers", type=str, default=None)\n    parser.add_argument("--contrast-bank", choices=tuple(CONTRAST_BANKS), default="proceed_hesitate")\n    args = parser.parse_args()\n\n    import torch\n    from transformers import AutoModelForCausalLM, AutoTokenizer\n\n    metadata = json.loads(args.direction_json.read_text(encoding="utf-8"))\n    semantic_direction = normalize(metadata["raw_delta"])\n    prompt_count = args.prompt_count if args.prompt_count is not None else (6 if args.mode == "smoke" else len(PROMPTS))\n    random_count = args.random_count if args.random_count is not None else (5 if args.mode == "smoke" else 63)\n    directions = [("semantic", 0, semantic_direction)]\n    directions.extend(\n        ("random", index, orthogonal_random_direction(semantic_direction, 20260703 + index))\n        for index in range(random_count)\n    )\n    layer_list = parse_int_list(args.layers)\n\n    device = "cuda" if torch.cuda.is_available() else "cpu"\n    dtype = torch.float16 if device == "cuda" else torch.float32\n    tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)\n    model = AutoModelForCausalLM.from_pretrained(\n        MODEL_NAME,\n        torch_dtype=dtype,\n        device_map="auto" if device == "cuda" else None,\n    )\n    if device == "cpu":\n        model.to(device)\n    model.eval()\n    layers = decoder_layers(model)\n    default_layer = int(metadata["decoder_layer_index"])\n    target_layers = layer_list if layer_list else [default_layer]\n    bank_token_ids = {\n        bank_name: (token_ids(tokenizer, positive), token_ids(tokenizer, negative))\n        for bank_name, (positive, negative) in CONTRAST_BANKS.items()\n    }\n    primary_contrast_field = f"logit_contrast_{args.contrast_bank}"\n\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    output_path = args.output_dir / f"intervention_{args.mode}.jsonl"\n    run_id = 0\n    with output_path.open("w", encoding="utf-8") as handle:\n        for target_layer in target_layers:\n            for direction_kind, direction_index, direction in directions:\n                alphas = SEMANTIC_ALPHAS if direction_kind == "semantic" else RANDOM_ALPHAS\n                for prompt_id in range(prompt_count):\n                    text = f"Task: {PROMPTS[prompt_id]}\\nResponse:"\n                    inputs = tokenizer(text, return_tensors="pt").to(model.device)\n                    prompt_len = int(inputs["input_ids"].shape[-1])\n                    for alpha in alphas:\n                        hook = SteeringHook(torch, direction, alpha)\n                        registered = layers[target_layer].register_forward_hook(hook)\n                        try:\n                            with torch.inference_mode():\n                                result = model(**inputs, use_cache=False, return_dict=True)\n                                logits = result.logits[0, -1, :].float()\n                                bank_logit_contrasts = {}\n                                for bank_name, (positive_ids, negative_ids) in bank_token_ids.items():\n                                    positive_logit = float(logits[positive_ids].mean().cpu())\n                                    negative_logit = float(logits[negative_ids].mean().cpu())\n                                    bank_logit_contrasts[bank_name] = positive_logit - negative_logit\n                                generated_text = ""\n                                features = {}\n                                if direction_kind == "semantic":\n                                    generated = model.generate(\n                                        **inputs,\n                                        do_sample=False,\n                                        max_new_tokens=12,\n                                        use_cache=False,\n                                        pad_token_id=tokenizer.eos_token_id,\n                                    )\n                                    generated_text = tokenizer.decode(\n                                        generated[0, prompt_len:], skip_special_tokens=True\n                                    ).strip()\n                                    for bank_name, (positive_words, negative_words) in CONTRAST_BANKS.items():\n                                        counts = keyword_counts(generated_text, positive_words, negative_words)\n                                        features[f"positive_word_count_{bank_name}"] = counts["positive_word_count"]\n                                        features[f"negative_word_count_{bank_name}"] = counts["negative_word_count"]\n                                else:\n                                    for bank_name in CONTRAST_BANKS:\n                                        features[f"positive_word_count_{bank_name}"] = 0\n                                        features[f"negative_word_count_{bank_name}"] = 0\n                        finally:\n                            registered.remove()\n                        run_id += 1\n                        row = {\n                            "run_id": run_id,\n                            "model": MODEL_NAME,\n                            "mode": args.mode,\n                            "direction_kind": direction_kind,\n                            "direction_index": direction_index,\n                            "target_layer": target_layer,\n                            "alpha": alpha,\n                            "prompt_id": prompt_id,\n                            "prompt": text,\n                            **{f"logit_contrast_{bank_name}": value for bank_name, value in bank_logit_contrasts.items()},\n                            "logit_contrast": bank_logit_contrasts[args.contrast_bank],\n                            "output": generated_text,\n                            **features,\n                        }\n                        handle.write(json.dumps(row, ensure_ascii=False) + "\\n")\n                        print(\n                            f"{run_id:04d} layer={target_layer:02d} {direction_kind}:{direction_index:02d} "\n                            f"prompt={prompt_id:02d} alpha={alpha:+.1f} "\n                            f"contrast={row[primary_contrast_field]:+.3f}"\n                        )\n    print(output_path)\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


In [ ]:
(WORK/'analyze_intervention.py').write_text('from __future__ import annotations\n\nimport argparse\nimport csv\nimport json\nfrom collections import defaultdict\nfrom pathlib import Path\n\nfrom intervention_core import exact_sign_flip_p, linear_slope\n\n\ndef load_rows(path: Path) -> list[dict]:\n    with path.open("r", encoding="utf-8") as handle:\n        return [json.loads(line) for line in handle if line.strip()]\n\n\ndef rows_for_layer(rows: list[dict], layer_index: int | None) -> list[dict]:\n    if layer_index is None:\n        return rows\n    return [row for row in rows if int(row["target_layer"]) == layer_index]\n\n\ndef contrast_field(bank_name: str) -> str:\n    return "logit_contrast" if bank_name == "action_calm" else f"logit_contrast_{bank_name}"\n\n\ndef positive_count_field(bank_name: str) -> str:\n    return "active_word_count" if bank_name == "action_calm" else f"positive_word_count_{bank_name}"\n\n\ndef negative_count_field(bank_name: str) -> str:\n    return "calm_word_count" if bank_name == "action_calm" else f"negative_word_count_{bank_name}"\n\n\ndef semantic_slopes(\n    rows: list[dict],\n    layer_index: int | None = None,\n    contrast_bank: str = "proceed_hesitate",\n) -> list[float]:\n    grouped = defaultdict(list)\n    for row in rows_for_layer(rows, layer_index):\n        if row["direction_kind"] == "semantic":\n            grouped[int(row["prompt_id"])].append(row)\n    slopes = []\n    field = contrast_field(contrast_bank)\n    for prompt_rows in grouped.values():\n        prompt_rows.sort(key=lambda row: float(row["alpha"]))\n        slopes.append(\n            linear_slope(\n                [float(row["alpha"]) for row in prompt_rows],\n                [float(row[field]) for row in prompt_rows],\n            )\n        )\n    return slopes\n\n\ndef random_slopes(\n    rows: list[dict],\n    layer_index: int | None = None,\n    contrast_bank: str = "proceed_hesitate",\n) -> dict[int, float]:\n    grouped = defaultdict(list)\n    for row in rows_for_layer(rows, layer_index):\n        if row["direction_kind"] == "random":\n            grouped[int(row["direction_index"])].append(row)\n    slopes = {}\n    field = contrast_field(contrast_bank)\n    for direction_index, direction_rows in grouped.items():\n        negative = [float(row[field]) for row in direction_rows if float(row["alpha"]) < 0]\n        positive = [float(row[field]) for row in direction_rows if float(row["alpha"]) > 0]\n        slopes[direction_index] = (sum(positive) / len(positive) - sum(negative) / len(negative)) / 4.0\n    return slopes\n\n\ndef random_distribution_rows(random_by_direction: dict[int, float], semantic_mean: float) -> list[dict[str, object]]:\n    ranked = sorted(random_by_direction.items(), key=lambda item: abs(item[1]), reverse=True)\n    return [\n        {\n            "rank": rank,\n            "direction_index": direction_index,\n            "random_logit_slope": slope,\n            "random_abs_logit_slope": abs(slope),\n            "beats_semantic_abs": abs(slope) >= abs(semantic_mean),\n        }\n        for rank, (direction_index, slope) in enumerate(ranked, start=1)\n    ]\n\n\ndef output_summary(\n    rows: list[dict],\n    layer_index: int | None = None,\n    contrast_bank: str = "proceed_hesitate",\n) -> list[dict[str, object]]:\n    grouped = defaultdict(list)\n    for row in rows_for_layer(rows, layer_index):\n        if row["direction_kind"] == "semantic":\n            grouped[float(row["alpha"])].append(row)\n    pos_field = positive_count_field(contrast_bank)\n    neg_field = negative_count_field(contrast_bank)\n    field = contrast_field(contrast_bank)\n    return [\n        {\n            "alpha": alpha,\n            "n": len(items),\n            "mean_logit_contrast": sum(float(row[field]) for row in items) / len(items),\n            "mean_positive_words": sum(int(row[pos_field]) for row in items) / len(items),\n            "mean_negative_words": sum(int(row[neg_field]) for row in items) / len(items),\n        }\n        for alpha, items in sorted(grouped.items())\n    ]\n\n\ndef metrics_for_layer(\n    rows: list[dict],\n    layer_index: int,\n    contrast_bank: str = "proceed_hesitate",\n) -> dict[str, object]:\n    sem_slopes = semantic_slopes(rows, layer_index=layer_index, contrast_bank=contrast_bank)\n    random_by_direction = random_slopes(rows, layer_index=layer_index, contrast_bank=contrast_bank)\n    rand_slopes = list(random_by_direction.values())\n    mean_semantic = sum(sem_slopes) / len(sem_slopes)\n    random_abs = sorted(abs(value) for value in rand_slopes)\n    semantic_abs = abs(mean_semantic)\n    random_beating_semantic = sum(value >= semantic_abs for value in random_abs)\n    random_quantile = (\n        sum(value <= semantic_abs for value in random_abs) / len(random_abs)\n        if random_abs\n        else None\n    )\n    return {\n        "layer_index": layer_index,\n        "contrast_bank": contrast_bank,\n        "prompt_count": len(sem_slopes),\n        "random_direction_count": len(rand_slopes),\n        "semantic_mean_logit_slope": mean_semantic,\n        "semantic_median_logit_slope": sorted(sem_slopes)[len(sem_slopes) // 2],\n        "semantic_sign_flip_p": exact_sign_flip_p(sem_slopes),\n        "random_abs_slope_mean": sum(abs(value) for value in rand_slopes) / len(rand_slopes) if rand_slopes else 0.0,\n        "random_abs_slope_median": random_abs[len(random_abs) // 2] if random_abs else 0.0,\n        "random_abs_slope_max": random_abs[-1] if random_abs else 0.0,\n        "random_directions_beating_semantic_abs": random_beating_semantic,\n        "semantic_abs_slope_random_quantile": random_quantile,\n        "random_specificity_rank_p": (\n            (1 + random_beating_semantic) / (len(rand_slopes) + 1)\n            if rand_slopes\n            else None\n        ),\n    }\n\n\ndef main() -> None:\n    parser = argparse.ArgumentParser()\n    parser.add_argument("--runs", type=Path, required=True)\n    parser.add_argument("--output-dir", type=Path, required=True)\n    parser.add_argument("--contrast-bank", choices=("action_calm", "proceed_hesitate"), default="proceed_hesitate")\n    args = parser.parse_args()\n    rows = load_rows(args.runs)\n    layers = sorted({int(row["target_layer"]) for row in rows})\n    args.output_dir.mkdir(parents=True, exist_ok=True)\n    if len(layers) == 1:\n        layer_index = layers[0]\n        result = metrics_for_layer(rows, layer_index, contrast_bank=args.contrast_bank)\n        result["mode"] = rows[0]["mode"]\n        summaries = output_summary(rows, layer_index=layer_index, contrast_bank=args.contrast_bank)\n        (args.output_dir / "causal_summary.json").write_text(\n            json.dumps(result, indent=2) + "\\n", encoding="utf-8"\n        )\n        with (args.output_dir / "alpha_summary.csv").open("w", encoding="utf-8", newline="") as handle:\n            writer = csv.DictWriter(handle, fieldnames=list(summaries[0]))\n            writer.writeheader()\n            writer.writerows(summaries)\n        random_rows = random_distribution_rows(\n            random_slopes(rows, layer_index=layer_index, contrast_bank=args.contrast_bank),\n            float(result["semantic_mean_logit_slope"]),\n        )\n        if random_rows:\n            with (args.output_dir / "random_slope_distribution.csv").open(\n                "w", encoding="utf-8", newline=""\n            ) as handle:\n                writer = csv.DictWriter(handle, fieldnames=list(random_rows[0]))\n                writer.writeheader()\n                writer.writerows(random_rows)\n        report = [\n            "# Qwen NeuroState Causal Intervention",\n            "",\n            f"- mode: `{result[\'mode\']}`",\n            f"- contrast bank: `{result[\'contrast_bank\']}`",\n            f"- layer: {result[\'layer_index\']}",\n            f"- prompts: {result[\'prompt_count\']}",\n            f"- random orthogonal directions: {result[\'random_direction_count\']}",\n            f"- semantic mean logit slope: {result[\'semantic_mean_logit_slope\']:.6f}",\n            f"- exact paired sign-flip p: {result[\'semantic_sign_flip_p\']:.6f}",\n            f"- random-direction specificity rank p: {result[\'random_specificity_rank_p\']:.6f}",\n            f"- random directions beating semantic abs slope: {result[\'random_directions_beating_semantic_abs\']}",\n            f"- semantic abs slope random quantile: {result[\'semantic_abs_slope_random_quantile\']:.6f}",\n            f"- max random abs slope: {result[\'random_abs_slope_max\']:.6f}",\n            "",\n            "| alpha | mean logit contrast | positive words | negative words |",\n            "|---:|---:|---:|---:|",\n        ]\n        for row in summaries:\n            report.append(\n                "| {alpha:+.1f} | {mean_logit_contrast:.4f} | "\n                "{mean_positive_words:.3f} | {mean_negative_words:.3f} |".format(**row)\n            )\n        if random_rows:\n            report.extend([\n                "",\n                "| random rank | direction | slope | abs slope | beats semantic |",\n                "|---:|---:|---:|---:|:---:|",\n            ])\n            for row in random_rows[:10]:\n                report.append(\n                    "| {rank} | {direction_index} | {random_logit_slope:.6f} | "\n                    "{random_abs_logit_slope:.6f} | {beats_semantic_abs} |".format(**row)\n                )\n        report.extend([\n            "",\n            "A causal claim requires both a non-zero paired slope and specificity relative to random directions.",\n            "The sign indicates orientation only; a reversed sign does not invalidate causal influence.",\n            "`random_slope_distribution.csv` lists every random direction ranked by absolute slope.",\n        ])\n        (args.output_dir / "causal_report.md").write_text("\\n".join(report) + "\\n", encoding="utf-8")\n        print(json.dumps(result, indent=2))\n        return\n\n    layer_results = [metrics_for_layer(rows, layer_index, contrast_bank=args.contrast_bank) for layer_index in layers]\n    result = {\n        "mode": rows[0]["mode"],\n        "contrast_bank": args.contrast_bank,\n        "layer_count": len(layer_results),\n        "layer_results": layer_results,\n    }\n    (args.output_dir / "causal_summary.json").write_text(\n        json.dumps(result, indent=2) + "\\n", encoding="utf-8"\n    )\n    with (args.output_dir / "layer_summary.csv").open("w", encoding="utf-8", newline="") as handle:\n        writer = csv.DictWriter(\n            handle,\n            fieldnames=[\n                "layer_index",\n                "contrast_bank",\n                "prompt_count",\n                "random_direction_count",\n                "semantic_mean_logit_slope",\n                "semantic_median_logit_slope",\n                "semantic_sign_flip_p",\n                "random_abs_slope_mean",\n                "random_abs_slope_median",\n                "random_abs_slope_max",\n                "random_directions_beating_semantic_abs",\n                "semantic_abs_slope_random_quantile",\n                "random_specificity_rank_p",\n            ],\n        )\n        writer.writeheader()\n        writer.writerows(layer_results)\n    report = [\n        "# Qwen NeuroState Causal Intervention",\n        "",\n        f"- mode: `{result[\'mode\']}`",\n        f"- contrast bank: `{result[\'contrast_bank\']}`",\n        f"- layers: {\', \'.join(str(layer) for layer in layers)}",\n        f"- layer_count: {result[\'layer_count\']}",\n        "",\n        "| layer | prompts | random dirs | mean slope | sign-flip p | specificity p |",\n        "|---:|---:|---:|---:|---:|---:|",\n    ]\n    for item in layer_results:\n        report.append(\n            "| {layer_index} | {prompt_count} | {random_direction_count} | "\n            "{semantic_mean_logit_slope:.6f} | {semantic_sign_flip_p:.6f} | "\n            "{random_specificity_rank_p:.6f} |".format(**item)\n        )\n    report.extend([\n        "",\n        "Layer sweeps are for locating where the signal lives; they do not by themselves establish specificity.",\n    ])\n    (args.output_dir / "causal_report.md").write_text("\\n".join(report) + "\\n", encoding="utf-8")\n    print(json.dumps(result, indent=2))\n\n\nif __name__ == "__main__":\n    main()\n',encoding='utf-8')


## Build v0.5 Direction

In [ ]:
!cd /content/neurostate_qwen_causal_intervention_v0_5 && python build_direction.py --target-layer 13 --output qwen_proceed_hesitate_direction_v0_5.json


## Layer 7 vs 13 Check

In [ ]:
!cd /content/neurostate_qwen_causal_intervention_v0_5 && python run_intervention.py --direction-json qwen_proceed_hesitate_direction_v0_5.json --mode layer_scan --layers 7,13 --prompt-count 8 --random-count 15 --contrast-bank proceed_hesitate --output-dir outputs_layer_compare


In [ ]:
!cd /content/neurostate_qwen_causal_intervention_v0_5 && python analyze_intervention.py --runs outputs_layer_compare/intervention_layer_scan.jsonl --contrast-bank proceed_hesitate --output-dir analysis_layer_compare


In [ ]:
from IPython.display import Markdown,display
display(Markdown((WORK/'analysis_layer_compare'/'causal_report.md').read_text()))


## Optional Layer 13 Full Run

Run this if layer 13 remains stronger than layer 7.

In [ ]:
# !cd /content/neurostate_qwen_causal_intervention_v0_5 && python run_intervention.py --direction-json qwen_proceed_hesitate_direction_v0_5.json --mode full --layers 13 --contrast-bank proceed_hesitate --output-dir outputs_full
# !cd /content/neurostate_qwen_causal_intervention_v0_5 && python analyze_intervention.py --runs outputs_full/intervention_full.jsonl --contrast-bank proceed_hesitate --output-dir analysis_full


In [ ]:
import shutil
from google.colab import files
archive=shutil.make_archive('/content/neurostate_qwen_causal_intervention_v0_5_results','zip',WORK)
files.download(archive)
